## 📚 **1. IMPORTACIÓN Y CONFIGURACIÓN**

In [ ]:
# Librerías básicas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import pickle
from datetime import datetime, timedelta

# Prophet y optimización
try:
    from prophet import Prophet
    prophet_available = True
    print("✅ Prophet importado correctamente")
except ImportError:
    try:
        from fbprophet import Prophet
        prophet_available = True
        print("✅ fbprophet importado correctamente")
    except ImportError:
        prophet_available = False
        print("❌ Prophet no está disponible. Instala con: pip install prophet")

from sklearn.metrics import mean_squared_error, mean_absolute_error
import optuna

# Configuraciones
warnings.filterwarnings('ignore')
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print(f"📅 Fecha de análisis: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

## 📂 **2. CARGA DE DATOS**

In [ ]:
# Cargar datos del análisis exploratorio
try:
    with open('datasets_preparados.pkl', 'rb') as f:
        datasets = pickle.load(f)
    
    # Extraer datasets
    data = datasets['data']
    train_to1 = datasets['train_to1']
    train_to2 = datasets['train_to2']
    test_t1 = datasets['test_t1']
    test_to2 = datasets['test_to2']
    train_len = datasets['train_len']
    
    print("✅ Datos cargados exitosamente")
    print(f"📊 Configuración: {train_len} train / {len(data) - train_len} test")
    
except FileNotFoundError:
    print("❌ Error: No se encontró 'datasets_preparados.pkl'")
    print("💡 Ejecuta primero el notebook '01_Analisis_Exploratorio.ipynb'")

if not prophet_available:
    print("\n⚠️ ADVERTENCIA: Prophet no está disponible")
    print("Para instalar Prophet:")
    print("   pip install prophet")
    print("   # o")
    print("   conda install -c conda-forge prophet")

## 🔧 **3. PREPARACIÓN DE DATOS PARA PROPHET**

In [ ]:
if prophet_available:
    # Prophet requiere columnas específicas: 'ds' (fecha) y 'y' (valor)
    
    # Crear fechas sintéticas (Prophet necesita fechas reales)
    start_date = datetime(2020, 1, 1)
    dates = [start_date + timedelta(days=i) for i in range(len(data))]
    
    # Preparar datos para Producto 1
    df_prophet_p1_train = pd.DataFrame({
        'ds': dates[:train_len],
        'y': train_to1['producto1'].values
    })
    
    df_prophet_p1_test = pd.DataFrame({
        'ds': dates[train_len:],
        'y': test_t1['producto1'].values
    })
    
    # Preparar datos para Producto 2
    df_prophet_p2_train = pd.DataFrame({
        'ds': dates[:train_len],
        'y': train_to2['producto2'].values
    })
    
    df_prophet_p2_test = pd.DataFrame({
        'ds': dates[train_len:],
        'y': test_to2['producto2'].values
    })
    
    print("✅ Datos preparados para Prophet")
    print(f"📊 Producto 1 - Train: {df_prophet_p1_train.shape}, Test: {df_prophet_p1_test.shape}")
    print(f"📊 Producto 2 - Train: {df_prophet_p2_train.shape}, Test: {df_prophet_p2_test.shape}")
    print(f"📅 Rango de fechas: {dates[0].strftime('%Y-%m-%d')} a {dates[-1].strftime('%Y-%m-%d')}")
else:
    print("⚠️ No se pueden preparar datos sin Prophet")

## 🔧 **4. FUNCIÓN DE EVALUACIÓN PROPHET**

In [ ]:
if prophet_available:
    def evaluar_prophet_rolling(df_train, df_test, **prophet_params):
        """
        Evalúa Prophet con validación temporal tipo walking forward
        
        Parameters:
        -----------
        df_train : pd.DataFrame
            Datos de entrenamiento con columnas 'ds' y 'y'
        df_test : pd.DataFrame
            Datos de prueba con columnas 'ds' y 'y'
        **prophet_params : dict
            Parámetros para Prophet
            
        Returns:
        --------
        float
            RMSE de las predicciones
        """
        try:
            # Configurar Prophet con parámetros
            model = Prophet(
                changepoint_prior_scale=prophet_params.get('changepoint_prior_scale', 0.05),
                seasonality_prior_scale=prophet_params.get('seasonality_prior_scale', 10.0),
                holidays_prior_scale=prophet_params.get('holidays_prior_scale', 10.0),
                seasonality_mode=prophet_params.get('seasonality_mode', 'additive'),
                yearly_seasonality=prophet_params.get('yearly_seasonality', 'auto'),
                weekly_seasonality=prophet_params.get('weekly_seasonality', 'auto'),
                daily_seasonality=prophet_params.get('daily_seasonality', False)
            )
            
            # Entrenar modelo
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                model.fit(df_train)
            
            # Crear dataframe para predicción
            future = model.make_future_dataframe(periods=len(df_test))
            
            # Generar predicciones
            forecast = model.predict(future)
            
            # Extraer predicciones para período de test
            pred_test = forecast['yhat'][-len(df_test):].values
            true_test = df_test['y'].values
            
            # Calcular RMSE
            rmse = np.sqrt(mean_squared_error(true_test, pred_test))
            
            return rmse
            
        except Exception as e:
            print(f"Error en Prophet: {str(e)}")
            return np.inf
    
    print("✅ Función de evaluación Prophet definida")
else:
    print("⚠️ Función Prophet no disponible")

## 🎯 **5. OPTIMIZACIÓN PROPHET - PRODUCTO 1**

In [ ]:
if prophet_available:
    def objective_prophet_p1(trial):
        """Función objetivo para optimización de Prophet - Producto 1"""
        params = {
            'changepoint_prior_scale': trial.suggest_float('changepoint_prior_scale', 0.001, 0.5),
            'seasonality_prior_scale': trial.suggest_float('seasonality_prior_scale', 0.01, 10),
            'holidays_prior_scale': trial.suggest_float('holidays_prior_scale', 0.01, 10),
            'seasonality_mode': trial.suggest_categorical('seasonality_mode', ['additive', 'multiplicative']),
            'yearly_seasonality': trial.suggest_categorical('yearly_seasonality', [True, False, 'auto']),
            'weekly_seasonality': trial.suggest_categorical('weekly_seasonality', [True, False, 'auto'])
        }
        
        return evaluar_prophet_rolling(df_prophet_p1_train, df_prophet_p1_test, **params)
    
    print("🔮 OPTIMIZACIÓN PROPHET - PRODUCTO 1")
    print("=" * 50)
    print("🚀 Iniciando optimización con Optuna...")
    print("⏱️ Esto puede tomar varios minutos...")
    print("-" * 60)
    
    study_prophet_p1 = optuna.create_study(
        direction="minimize",
        pruner=optuna.pruners.MedianPruner()
    )
    
    study_prophet_p1.optimize(objective_prophet_p1, n_trials=50, timeout=900)  # 15 min timeout
    
    print("\n" + "=" * 60)
    print("RESULTADOS OPTIMIZACIÓN PROPHET - PRODUCTO 1")
    print("=" * 60)
    print(f"Mejor RMSE: {study_prophet_p1.best_value:.4f}")
    print(f"\nMejores parámetros:")
    for param, value in study_prophet_p1.best_params.items():
        if isinstance(value, float):
            print(f"  {param}: {value:.4f}")
        else:
            print(f"  {param}: {value}")
    print(f"\nTrials completados: {len(study_prophet_p1.trials)}")
    print("=" * 60)
    
else:
    print("⚠️ Prophet no disponible - saltando optimización Producto 1")
    study_prophet_p1 = None

## 📊 **6. MODELO FINAL PRODUCTO 1**

In [ ]:
if prophet_available and study_prophet_p1 is not None:
    # Entrenar modelo final con mejores parámetros
    best_params_p1 = study_prophet_p1.best_params
    
    print("📊 ENTRENAMIENTO MODELO FINAL - PRODUCTO 1")
    print("=" * 50)
    
    modelo_prophet_p1 = Prophet(
        changepoint_prior_scale=best_params_p1['changepoint_prior_scale'],
        seasonality_prior_scale=best_params_p1['seasonality_prior_scale'],
        holidays_prior_scale=best_params_p1['holidays_prior_scale'],
        seasonality_mode=best_params_p1['seasonality_mode'],
        yearly_seasonality=best_params_p1['yearly_seasonality'],
        weekly_seasonality=best_params_p1['weekly_seasonality'],
        daily_seasonality=False
    )
    
    # Entrenar con datos completos
    df_completo_p1 = pd.concat([df_prophet_p1_train, df_prophet_p1_test])
    
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        modelo_prophet_p1.fit(df_completo_p1)
    
    print("✅ Modelo entrenado exitosamente")
    
    # Generar pronósticos futuros
    future_p1 = modelo_prophet_p1.make_future_dataframe(periods=12)  # 12 períodos futuros
    forecast_p1 = modelo_prophet_p1.predict(future_p1)
    
    print(f"🔮 Pronósticos generados para 12 períodos futuros")
    print(f"📈 Último pronóstico: {forecast_p1['yhat'].iloc[-1]:.2f}")
    
else:
    print("⚠️ No se puede entrenar modelo Prophet Producto 1")
    modelo_prophet_p1 = None
    forecast_p1 = None

## 🎯 **7. OPTIMIZACIÓN PROPHET - PRODUCTO 2**

In [ ]:
if prophet_available:
    def objective_prophet_p2(trial):
        """Función objetivo para optimización de Prophet - Producto 2"""
        params = {
            'changepoint_prior_scale': trial.suggest_float('changepoint_prior_scale', 0.001, 0.5),
            'seasonality_prior_scale': trial.suggest_float('seasonality_prior_scale', 0.01, 10),
            'holidays_prior_scale': trial.suggest_float('holidays_prior_scale', 0.01, 10),
            'seasonality_mode': trial.suggest_categorical('seasonality_mode', ['additive', 'multiplicative']),
            'yearly_seasonality': trial.suggest_categorical('yearly_seasonality', [True, False, 'auto']),
            'weekly_seasonality': trial.suggest_categorical('weekly_seasonality', [True, False, 'auto'])
        }
        
        return evaluar_prophet_rolling(df_prophet_p2_train, df_prophet_p2_test, **params)
    
    print("🔮 OPTIMIZACIÓN PROPHET - PRODUCTO 2")
    print("=" * 50)
    print("🚀 Iniciando optimización...")
    print("-" * 60)
    
    study_prophet_p2 = optuna.create_study(
        direction="minimize",
        pruner=optuna.pruners.MedianPruner()
    )
    
    study_prophet_p2.optimize(objective_prophet_p2, n_trials=50, timeout=900)
    
    print("\n" + "=" * 60)
    print("RESULTADOS OPTIMIZACIÓN PROPHET - PRODUCTO 2")
    print("=" * 60)
    print(f"Mejor RMSE: {study_prophet_p2.best_value:.4f}")
    print(f"\nMejores parámetros:")
    for param, value in study_prophet_p2.best_params.items():
        if isinstance(value, float):
            print(f"  {param}: {value:.4f}")
        else:
            print(f"  {param}: {value}")
    print(f"\nTrials completados: {len(study_prophet_p2.trials)}")
    print("=" * 60)
    
else:
    print("⚠️ Prophet no disponible - saltando optimización Producto 2")
    study_prophet_p2 = None

## 📊 **8. MODELO FINAL PRODUCTO 2**

In [ ]:
if prophet_available and study_prophet_p2 is not None:
    # Entrenar modelo final Producto 2
    best_params_p2 = study_prophet_p2.best_params
    
    print("📊 ENTRENAMIENTO MODELO FINAL - PRODUCTO 2")
    print("=" * 50)
    
    modelo_prophet_p2 = Prophet(
        changepoint_prior_scale=best_params_p2['changepoint_prior_scale'],
        seasonality_prior_scale=best_params_p2['seasonality_prior_scale'],
        holidays_prior_scale=best_params_p2['holidays_prior_scale'],
        seasonality_mode=best_params_p2['seasonality_mode'],
        yearly_seasonality=best_params_p2['yearly_seasonality'],
        weekly_seasonality=best_params_p2['weekly_seasonality'],
        daily_seasonality=False
    )
    
    df_completo_p2 = pd.concat([df_prophet_p2_train, df_prophet_p2_test])
    
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        modelo_prophet_p2.fit(df_completo_p2)
    
    # Generar pronósticos
    future_p2 = modelo_prophet_p2.make_future_dataframe(periods=12)
    forecast_p2 = modelo_prophet_p2.predict(future_p2)
    
    print("✅ Modelo entrenado exitosamente")
    print(f"🔮 Pronósticos generados para 12 períodos futuros")
    print(f"📈 Último pronóstico: {forecast_p2['yhat'].iloc[-1]:.2f}")
    
else:
    print("⚠️ No se puede entrenar modelo Prophet Producto 2")
    modelo_prophet_p2 = None
    forecast_p2 = None

## 📈 **9. VISUALIZACIONES PROPHET**

In [ ]:
if prophet_available and forecast_p1 is not None and forecast_p2 is not None:
    # Visualizar pronósticos Prophet
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 12))
    
    # Producto 1
    modelo_prophet_p1.plot(forecast_p1, ax=ax1)
    ax1.set_title('🔮 Pronósticos Prophet - Producto 1', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Valores')
    ax1.grid(True, alpha=0.3)
    
    # Producto 2
    modelo_prophet_p2.plot(forecast_p2, ax=ax2)
    ax2.set_title('🔮 Pronósticos Prophet - Producto 2', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Valores')
    ax2.set_xlabel('Fecha')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Visualizar componentes
    print("\n📊 COMPONENTES DEL MODELO PROPHET")
    print("=" * 50)
    
    # Componentes Producto 1
    print("📦 Componentes Producto 1:")
    fig1 = modelo_prophet_p1.plot_components(forecast_p1)
    fig1.suptitle('📊 Componentes Prophet - Producto 1', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Componentes Producto 2
    print("\n📦 Componentes Producto 2:")
    fig2 = modelo_prophet_p2.plot_components(forecast_p2)
    fig2.suptitle('📊 Componentes Prophet - Producto 2', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
else:
    print("⚠️ No se pueden generar visualizaciones Prophet")

## 📊 **10. ANÁLISIS COMPARATIVO**

In [ ]:
if prophet_available and study_prophet_p1 is not None and study_prophet_p2 is not None:
    # Comparar con resultados anteriores
    print("📊 RESUMEN COMPARATIVO - TODOS LOS MODELOS")
    print("=" * 70)
    
    # Cargar resultados anteriores si existen
    try:
        with open('resultados_ma_hw.pkl', 'rb') as f:
            resultados_ma_hw = pickle.load(f)
        ma_hw_disponible = True
    except:
        ma_hw_disponible = False
        
    try:
        with open('resultados_arima.pkl', 'rb') as f:
            resultados_arima = pickle.load(f)
        arima_disponible = True
    except:
        arima_disponible = False
    
    # Tabla comparativa
    modelos = ['Prophet']
    rmse_p1 = [study_prophet_p1.best_value]
    rmse_p2 = [study_prophet_p2.best_value]
    
    if ma_hw_disponible:
        # Agregar mejores resultados MA/HW
        mejor_ma_p1 = min(resultados_ma_hw['ma_p1'].items(), key=lambda x: x[1]['rmse'])
        mejor_ma_p2 = min(resultados_ma_hw['ma_p2'].items(), key=lambda x: x[1]['rmse'])
        
        modelos.extend([f'Media Móvil ({mejor_ma_p1[0]}/{mejor_ma_p2[0]})', 'Holt-Winters'])
        rmse_p1.extend([mejor_ma_p1[1]['rmse'], resultados_ma_hw['hw_p1']['best_rmse']])
        rmse_p2.extend([mejor_ma_p2[1]['rmse'], resultados_ma_hw['hw_p2']['best_rmse']])
    
    if arima_disponible:
        modelos.append('ARIMA')
        rmse_p1.append(resultados_arima['p1']['best_rmse'])
        rmse_p2.append(resultados_arima['p2']['best_rmse'])
    
    # Crear DataFrame comparativo
    df_comparacion = pd.DataFrame({
        'Modelo': modelos,
        'RMSE Producto 1': rmse_p1,
        'RMSE Producto 2': rmse_p2
    })
    
    # Agregar ranking
    df_comparacion['Ranking P1'] = df_comparacion['RMSE Producto 1'].rank()
    df_comparacion['Ranking P2'] = df_comparacion['RMSE Producto 2'].rank()
    
    print("🏆 TABLA COMPARATIVA DE MODELOS:")
    print(df_comparacion.round(4).to_string(index=False))
    
    # Mejores modelos
    mejor_p1_idx = df_comparacion['RMSE Producto 1'].idxmin()
    mejor_p2_idx = df_comparacion['RMSE Producto 2'].idxmin()
    
    print(f"\n🥇 MEJORES MODELOS:")
    print(f"   • Producto 1: {df_comparacion.loc[mejor_p1_idx, 'Modelo']} (RMSE: {df_comparacion.loc[mejor_p1_idx, 'RMSE Producto 1']:.4f})")
    print(f"   • Producto 2: {df_comparacion.loc[mejor_p2_idx, 'Modelo']} (RMSE: {df_comparacion.loc[mejor_p2_idx, 'RMSE Producto 2']:.4f})")
    
else:
    print("⚠️ No se puede realizar análisis comparativo completo")

## 💾 **11. GUARDAR RESULTADOS FINALES**

In [ ]:
if prophet_available:
    # Guardar resultados Prophet
    resultados_prophet = {
        'p1': {
            'study': study_prophet_p1 if 'study_prophet_p1' in locals() else None,
            'best_params': study_prophet_p1.best_params if 'study_prophet_p1' in locals() and study_prophet_p1 is not None else None,
            'best_rmse': study_prophet_p1.best_value if 'study_prophet_p1' in locals() and study_prophet_p1 is not None else None,
            'modelo': modelo_prophet_p1 if 'modelo_prophet_p1' in locals() else None,
            'forecast': forecast_p1 if 'forecast_p1' in locals() else None
        },
        'p2': {
            'study': study_prophet_p2 if 'study_prophet_p2' in locals() else None,
            'best_params': study_prophet_p2.best_params if 'study_prophet_p2' in locals() and study_prophet_p2 is not None else None,
            'best_rmse': study_prophet_p2.best_value if 'study_prophet_p2' in locals() and study_prophet_p2 is not None else None,
            'modelo': modelo_prophet_p2 if 'modelo_prophet_p2' in locals() else None,
            'forecast': forecast_p2 if 'forecast_p2' in locals() else None
        }
    }
    
    with open('resultados_prophet.pkl', 'wb') as f:
        pickle.dump(resultados_prophet, f)
    
    print("💾 Resultados Prophet guardados en 'resultados_prophet.pkl'")
    
    # Resumen final
    print(f"\n📊 RESUMEN FINAL PROPHET:")
    if study_prophet_p1 is not None:
        print(f"   • Producto 1 RMSE: {study_prophet_p1.best_value:.4f}")
    if study_prophet_p2 is not None:
        print(f"   • Producto 2 RMSE: {study_prophet_p2.best_value:.4f}")
        
else:
    print("⚠️ Prophet no disponible - no se guardan resultados")

print("\n" + "=" * 70)
print("🚀 ANÁLISIS PROPHET COMPLETADO")
print("\n📝 NOTEBOOKS DISPONIBLES:")
print("   • 01_Analisis_Exploratorio.ipynb")
print("   • 02_Media_Movil_HoltWinters.ipynb")
print("   • 03_ARIMA.ipynb")
print("   • 04_Prophet.ipynb (actual)")
print("\n💡 Para análisis completo, ejecuta todos los notebooks en orden")
print("=" * 70)

## 📋 **12. CONCLUSIONES GENERALES**

In [ ]:
print("📋 CONCLUSIONES DEL ANÁLISIS COMPLETO")
print("=" * 70)

print("\n🎯 METODOLOGÍA IMPLEMENTADA:")
print("   ✅ Análisis exploratorio completo")
print("   ✅ Media móvil con validación temporal")
print("   ✅ Holt-Winters optimizado")
print("   ✅ ARIMA con validación de supuestos")
print("   ✅ Prophet con componentes automáticos")

print("\n📊 CARACTERÍSTICAS TÉCNICAS:")
print("   • Validación temporal (walking forward)")
print("   • Optimización bayesiana con Optuna")
print("   • Análisis de supuestos estadísticos")
print("   • Pronósticos con intervalos de confianza")
print("   • Visualizaciones interpretables")

print("\n🔄 MODELOS COMPARADOS:")
print("   1. Media Móvil Simple")
print("   2. Holt-Winters Exponential Smoothing")
print("   3. ARIMA (AutoRegressive Integrated Moving Average)")
print("   4. Facebook Prophet")

if prophet_available:
    print("\n✅ ANÁLISIS COMPLETADO EXITOSAMENTE")
    print("📈 Todos los modelos han sido evaluados y comparados")
else:
    print("\n⚠️ ANÁLISIS PARCIALMENTE COMPLETADO")
    print("📝 Prophet requiere instalación adicional")

print("\n💡 RECOMENDACIONES:")
print("   • Revisar la tabla comparativa de modelos")
print("   • Considerar el contexto de negocio para selección")
print("   • Validar pronósticos con datos reales futuros")
print("   • Monitorear rendimiento en producción")

print("\n" + "=" * 70)
print("🎉 ¡ANÁLISIS DE SERIES DE TIEMPO FINALIZADO!")
print("=" * 70)